# Sleep dashboard walkthrough
This notebook is a guided demo of the eywa_trees dashboard on the Sleep Health and Lifestyle dataset. It is documentation-style: we load the data once, then show how to explore the tool for both a regression task and a classification task.


In [ ]:
import threading

import pandas as pd

from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor
from sklearn.model_selection import train_test_split

from eywa_trees import SplitDash

seed = 7


def start_dashboard(app, port: int):
    thread = threading.Thread(target=lambda: app.run(port=port, debug=False), daemon=True)
    thread.start()
    print(f"Dashboard available at http://127.0.0.1:{port}")
    return thread



## Regression: predict nightly hours
We use `sleep-duration` as the target, drop the survey score `quality-of-sleep`, one-hot encode the remaining categoricals, and train a small random forest. The dashboard runs in a background thread so you can keep scrolling.


In [ ]:
df = pd.read_csv('data/DengueDatabase.csv',header=0)

df.drop('Mgrd01', axis=1, inplace=True)  ## O município foi atingido pela seca nos últimos 4 anos
df.drop('Mgrd07', axis=1, inplace=True)  ## O município foi atingido por processo erosivo acelerado nos últimos 4 anos
df.drop('Mgrd14', axis=1, inplace=True)  ## O município foi atingido por escorregamentos ou deslizamentos de encostas nos últimos 4 anos
df.drop('Mgrd181', axis=1, inplace=True) ## Mapeamentos de áreas de risco de enchentes ou inundações
#As variáveis Mgrd07,Mgrd14,Mgrd181 não farão parte do nosso estudo, tem outras variáveis com mais impo

#Unnamed: 0 # Não é uma variável. Vamos retirar essa coluna.
df.drop('Unnamed: 0', axis=1, inplace=True)

df.head()

seed = 123
class_target = "Surto"

df = df.dropna(axis=0, how='any')
X = df.drop(columns=[class_target])
cat_cols = X.select_dtypes(include=["object", "category"]).columns
X = pd.get_dummies(X, columns=cat_cols, prefix_sep="_is_", dtype=int)
y = df[class_target]
class_names = sorted(y.unique())

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=seed, stratify=y
)



In [ ]:
model = RandomForestClassifier(
    n_estimators=60,
    max_depth=6,
    min_samples_leaf=4,
    random_state=seed,
)
model.fit(X_train, y_train)

sdash = SplitDash(
    model,
    X_train,
    X_test,
    y_test,
    feature_names=X.columns,
    class_names=class_names,
)
sdash.config(show_text=True)

start_dashboard(sdash, port=8060)